# Spherical Fourier-Bessel $C_\ell(k_\parallel)$ Estimator — Results

This notebook presents the results of the sFB power spectrum estimator for the Ly-$\alpha$ forest.
We demonstrate:

1. **Multi-$k_\parallel$** recovery: The estimator recovers the input $P(k)$ at 10 independent
   $k_\parallel$ values, not just $k_\parallel=0$.
2. **RSD**: Redshift-space distortions via the Kaiser factor $(1+\beta\mu^2)^2$ are correctly
   recovered at each $k_\parallel$.
3. **Noise**: Adding 10% RMS Gaussian pixel noise and analytically subtracting the noise bias
   $N_\ell = N_{\rm pix}\sigma_c^2 N_{\rm skew}/(4\pi)$ yields results indistinguishable from
   the noiseless case.
4. **Mode marginalization**: Subtracting the per-LOS mean ($\delta \to \delta - \langle\delta\rangle$)
   perfectly removes the $k_\parallel=0$ mode without affecting any $k_\parallel \neq 0$ modes,
   providing a clean prescription for continuum-fitting systematics.

### Notation

| Symbol | Meaning |
|---|---|
| $C_\ell(k_n)$ | Angular power spectrum at LOS Fourier mode $k_n$ |
| $\tilde{C}_\ell(k_n)$ | Pseudo (measured) angular power spectrum |
| $M_{\ell L}$ | MASTER angular coupling matrix (from window function) |
| $k_\perp = (\ell+1/2)/\chi_{\rm eff}$ | Transverse wavenumber (Limber) |
| $\mu = k_\parallel / |\mathbf{k}|$ | Cosine of angle to LOS |

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Project paths
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root not in sys.path:
    sys.path.insert(0, root)
    sys.path.insert(0, os.path.join(root, 'notebooks'))

import matplotlib_params_file  # noqa

# Configure paths — edit these to point to your results
RESULTS_NOISELESS = os.path.join(root, 'results_multik_prod',
    'Cell_multik_L1380_N512_Nq9603_Nl500_Nk10_sims20_rsd_theory.npz')
RESULTS_NOISY = os.path.join(root, 'results_multik_prod',
    'Cell_multik_L1380_N512_Nq9603_Nl500_Nk10_sims20_noise0.10_rsd_theory.npz')
RESULTS_MARG = os.path.join(root, 'results_marginalization',
    'marginalization_test_L1380_N512_Nq9603_Nl200_Nk5_sims5_rsd.npz')

print(f'Noiseless: {os.path.exists(RESULTS_NOISELESS)}')
print(f'Noisy:     {os.path.exists(RESULTS_NOISY)}')
print(f'Marg test: {os.path.exists(RESULTS_MARG)}')

## 1. Load Results

In [ ]:
def load_theory(path):
    """Load a theory npz and return a dict with key arrays."""
    d = np.load(path)
    out = {k: d[k] for k in d.files}
    # Scalar conversions
    for k in ['Nl', 'Nl_large', 'Nskew', 'N', 'Nsims', 'NperBin']:
        if k in out:
            out[k] = int(out[k])
    for k in ['chi_eff', 'chi_shift', 'L_box', 'Lbox_3d', 'bias', 'beta',
              'W_floor', 'sigma_c', 'N_noise', 'noise_frac']:
        if k in out:
            out[k] = float(out[k])
    return out

dn = load_theory(RESULTS_NOISELESS)
dy = load_theory(RESULTS_NOISY)

print(f"Noiseless: {dn['Nsims']} sims, Nk={len(dn['k_par'])}, Nl={dn['Nl']}")
print(f"  bias={dn['bias']:.4f}, beta={dn['beta']:.4f}")
print(f"  k_par = {dn['k_par'][:5]} ...")
print(f"\nNoisy: sigma_c={dy['sigma_c']:.2f}, N_noise={dy['N_noise']:.2e}")

## 2. Pseudo-$C_\ell(k)$: Data vs Theory

The pseudo-power spectrum at each $k_\parallel$ should match the MASTER prediction:

$$
\langle \tilde{C}_\ell(k_n) \rangle = \sum_L M_{\ell L}^{\rm ang} \, C^{\rm true}(L, k_n) + C^{\rm floor}(k_n)
$$

In [ ]:
def make_binning_matrix(Nbins, NperBin, Nl):
    B = np.zeros((Nbins, Nl))
    for i in range(Nbins):
        lo = i * NperBin
        hi = min(lo + NperBin, Nl)
        B[i, lo:hi] = 1.0 / (hi - lo)
    return B

def plot_pseudo_cl(data, title_extra=''):
    k_par = data['k_par']
    ells = data['ells']
    Nk = len(k_par)
    Nl = data['Nl']
    NperBin = data['NperBin']
    binned_ells = data['binned_ells']
    Nbins = len(binned_ells)
    B = make_binning_matrix(Nbins, NperBin, Nl)
    Nsims = data['Nsims']
    cl_k_all = data['cl_k_all']
    cl_mean = data['cl_mean_all']
    theory_pseudo = data['theory_pseudo_all']

    ncols = min(3, Nk)
    nrows = (Nk + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 5*nrows),
                             squeeze=False, sharex=True)
    fig.subplots_adjust(hspace=0.3, wspace=0.35)
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, Nk))

    for ik in range(Nk):
        ir, ic = divmod(ik, ncols)
        ax = axes[ir, ic]
        binned_raw = B @ cl_mean[ik]
        binned_std = B @ (np.std(cl_k_all[:, ik, :], axis=0) / np.sqrt(Nsims))
        binned_theory = B @ theory_pseudo[ik]

        ax.errorbar(binned_ells, binned_raw, yerr=binned_std,
                    fmt='o', color=colors[ik], ms=4, capsize=2,
                    label=f'Measured ({Nsims} sims)')
        ax.plot(binned_ells, binned_theory, 'k--', lw=1.5, label='Theory')
        ax.set_title(f'$k_\\parallel = {k_par[ik]:.4f}$ h/Mpc')
        if ir == nrows - 1:
            ax.set_xlabel(r'$\ell$')
        ax.set_ylabel(r'pseudo-$C_\ell(k)$')
        ax.legend(fontsize=12, loc='upper right')

    for ik in range(Nk, nrows * ncols):
        ir, ic = divmod(ik, ncols)
        axes[ir, ic].set_visible(False)

    fig.suptitle(f'Pseudo-$C_\\ell(k)$: {Nsims} sims{title_extra}',
                 fontsize=20, y=1.01)
    plt.tight_layout()
    return fig

fig = plot_pseudo_cl(dn, ', noiseless + RSD')
plt.show()

## 3. Money Plot: Ratio (meas / theory) at All $k_\parallel$

The ratio $\tilde{C}_\ell^{\rm meas}(k_n) / \tilde{C}_\ell^{\rm theory}(k_n)$
should scatter around **1.0** for all $k_\parallel$ and $\ell$.

In [ ]:
def plot_ratio(data, title_extra=''):
    k_par = data['k_par']
    Nk = len(k_par)
    Nl = data['Nl']
    NperBin = data['NperBin']
    binned_ells = data['binned_ells']
    Nbins = len(binned_ells)
    B = make_binning_matrix(Nbins, NperBin, Nl)
    Nsims = data['Nsims']
    cl_k_all = data['cl_k_all']
    cl_mean = data['cl_mean_all']
    theory_pseudo = data['theory_pseudo_all']
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, Nk))

    fig = plt.figure(figsize=(14, 8))
    gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1)

    for ik in range(Nk):
        binned_raw = B @ cl_mean[ik]
        binned_std = B @ (np.std(cl_k_all[:, ik, :], axis=0) / np.sqrt(Nsims))
        binned_theory = B @ theory_pseudo[ik]
        offset = ik * 1.0

        ax1.errorbar(binned_ells + offset, binned_raw, yerr=binned_std,
                     fmt='o', color=colors[ik], ms=3, capsize=2,
                     label=f'$k_\\parallel$={k_par[ik]:.3f}')

        mask = binned_theory > 0
        ratio = np.where(mask, binned_raw / binned_theory, np.nan)
        ratio_err = np.where(mask, binned_std / binned_theory, np.nan)
        ax2.errorbar(binned_ells[mask] + offset, ratio[mask],
                     yerr=ratio_err[mask],
                     fmt='o', color=colors[ik], ms=3, capsize=1)

    # k=0 theory reference
    binned_theory_k0 = B @ theory_pseudo[0]
    ax1.plot(binned_ells, binned_theory_k0, 'k--', lw=2, label='Theory ($k$=0)')

    ax1.set_ylabel(r'binned pseudo-$C_\ell(k)$')
    ax1.legend(fontsize=11, ncol=min(4, Nk+1), loc='upper right')
    ax1.set_title(f'Multi-$k$ pseudo-$C_\\ell$: {Nsims} sims{title_extra}')
    ax1.tick_params(labelbottom=False)

    ax2.axhline(1, color='k', ls='--', lw=0.8)
    ax2.axhspan(0.95, 1.05, color='gray', alpha=0.15)
    ax2.set_xlabel(r'multipole $\ell$')
    ax2.set_ylabel('meas / theory')
    ax2.set_ylim(0.7, 1.3)
    plt.tight_layout()
    return fig

fig = plot_ratio(dn, ', noiseless + RSD')
plt.show()

## 4. Deconvolved $C_\ell(k)$ and Kaiser Factor

After floor subtraction and MASTER deconvolution, we recover the true angular power spectrum:

$$
C_\ell^{\rm true}(k_\parallel) = \frac{b^2(1+\beta\mu^2)^2 \, P_{\rm lin}(|\mathbf{k}|)}{L_{\rm box} \, \chi_{\rm eff}^2}
$$

where $|\mathbf{k}| = \sqrt{k_\perp^2 + k_\parallel^2}$ and $\mu = k_\parallel / |\mathbf{k}|$.

The fan of curves at different $k_\parallel$ shows the Kaiser enhancement:
higher $k_\parallel$ → larger $\mu$ → stronger RSD boost.

In [ ]:
def plot_deconv(data, title_extra=''):
    k_par = data['k_par']
    Nk = len(k_par)
    ells_dec = data['ells_dec']
    dec_mean = data['dec_mean']
    dec_std = data['dec_std']
    theory_dec = data['theory_dec_all']
    Nsims = data['Nsims']
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, Nk))

    fig = plt.figure(figsize=(14, 9))
    gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1)

    for ik in range(Nk):
        offset = ik * 1.0
        ax1.errorbar(ells_dec + offset, dec_mean[ik], yerr=dec_std[ik],
                     fmt='o', color=colors[ik], ms=3, capsize=2,
                     label=f'$k_\\parallel$={k_par[ik]:.3f}')
        ax1.plot(ells_dec, theory_dec[ik], '--', color=colors[ik], lw=1.5)

        mask = theory_dec[ik] > 0
        ratio = np.where(mask, dec_mean[ik] / theory_dec[ik], np.nan)
        ratio_err = np.where(mask, dec_std[ik] / theory_dec[ik], np.nan)
        ax2.errorbar(ells_dec[mask] + offset, ratio[mask],
                     yerr=ratio_err[mask],
                     fmt='o', color=colors[ik], ms=3, capsize=1)

    ax1.set_ylabel(r'deconvolved $C_\ell(k)$')
    ax1.legend(fontsize=11, ncol=min(4, Nk), loc='upper right')
    ax1.set_title(f'Floor-subtracted deconvolved $C_\\ell(k)$: '
                  f'{Nsims} sims{title_extra}')
    ax1.tick_params(labelbottom=False)

    ax2.axhline(1, color='k', ls='--', lw=0.8)
    ax2.axhspan(0.95, 1.05, color='gray', alpha=0.15)
    ax2.set_xlabel(r'multipole $\ell$')
    ax2.set_ylabel('meas / theory')
    ax2.set_ylim(0.7, 1.3)
    plt.tight_layout()
    return fig

fig = plot_deconv(dn, ', noiseless + RSD')
plt.show()

## 5. Summary Table: Mean Ratios per $k_\parallel$

In [ ]:
def summary_table(data, label=''):
    k_par = data['k_par']
    Nk = len(k_par)
    Nl = data['Nl']
    NperBin = data['NperBin']
    binned_ells = data['binned_ells']
    Nbins = len(binned_ells)
    B = make_binning_matrix(Nbins, NperBin, Nl)
    cl_mean = data['cl_mean_all']
    theory_pseudo = data['theory_pseudo_all']
    dec_mean = data['dec_mean']
    theory_dec = data['theory_dec_all']

    print(f'\n{label}')
    print(f'{"k_par":>10s} {"pseudo ratio":>14s} {"deconv ratio":>14s}')
    print('-' * 40)
    for ik in range(Nk):
        binned_raw = B @ cl_mean[ik]
        binned_theory = B @ theory_pseudo[ik]
        mask_p = binned_theory > 0
        rp = binned_raw[mask_p] / binned_theory[mask_p]

        mask_d = theory_dec[ik] > 0
        rd = dec_mean[ik][mask_d] / theory_dec[ik][mask_d]

        rp_m = np.mean(rp[1:]) if len(rp) > 1 else np.nan
        rd_m = np.mean(rd[1:]) if len(rd) > 1 else np.nan
        print(f'{k_par[ik]:10.5f} {rp_m:14.4f} {rd_m:14.4f}')

summary_table(dn, 'NOISELESS + RSD:')
summary_table(dy, 'NOISY (σ_c=0.10) + RSD:')

## 6. Noise Test: Noiseless vs Noisy (After Bias Subtraction)

Adding per-pixel Gaussian noise $N(0, \sigma_c^2)$ adds a white (ℓ-independent, k-independent)
bias that is analytically calculable:

$$
N_\ell = N_{\rm pix} \, \sigma_c^2 \, N_{\rm skew} / (4\pi)
$$

After subtraction, the noisy results should be indistinguishable from the noiseless case.

In [ ]:
k_par = dn['k_par']
Nk = len(k_par)
Nl = dn['Nl']
NperBin = dn['NperBin']
binned_ells = dn['binned_ells']
Nbins = len(binned_ells)
B = make_binning_matrix(Nbins, NperBin, Nl)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.05})
ax1, ax2 = axes
colors = plt.cm.viridis(np.linspace(0.1, 0.9, Nk))

for ik in [0, 4, 9]:  # sample k values
    # Noiseless
    bn = B @ dn['cl_mean_all'][ik]
    bt = B @ dn['theory_pseudo_all'][ik]
    mask = bt > 0
    ratio_n = np.where(mask, bn / bt, np.nan)

    # Noisy
    by = B @ dy['cl_mean_all'][ik]
    bty = B @ dy['theory_pseudo_all'][ik]
    ratio_y = np.where(mask, by / bty, np.nan)

    offset = ik * 0.5
    ax1.plot(binned_ells + offset, ratio_n[mask], 'o', color=colors[ik],
             ms=5, label=f'k={k_par[ik]:.3f} (clean)')
    ax1.plot(binned_ells + offset + 2, ratio_y[mask], 's', color=colors[ik],
             ms=5, mfc='none', mew=1.5, label=f'k={k_par[ik]:.3f} (noisy)')

    # Difference
    diff = ratio_y[mask] - ratio_n[mask]
    ax2.plot(binned_ells[mask] + offset, diff, 'o', color=colors[ik], ms=4)

ax1.axhline(1, color='k', ls='--', lw=0.8)
ax1.axhspan(0.95, 1.05, color='gray', alpha=0.12)
ax1.set_ylabel('meas / theory')
ax1.legend(fontsize=10, ncol=3, loc='upper right')
ax1.set_title(f'Noise test: clean (filled) vs $\\sigma_c$={dy["sigma_c"]:.2f} '
              f'(open, $N_\\ell$={dy["N_noise"]:.0f} subtracted)')
ax1.set_ylim(0.85, 1.15)

ax2.axhline(0, color='k', ls='--', lw=0.8)
ax2.set_xlabel(r'multipole $\ell$')
ax2.set_ylabel(r'$\Delta$(ratio)')
ax2.set_ylim(-0.05, 0.05)
plt.tight_layout()
plt.show()

## 7. Marginalization: $k_\parallel = 0$ Mode Removal

### The problem

Continuum-fitting errors add a constant offset to each line of sight, contaminating the
$k_\parallel = 0$ mode. The standard fix is to subtract the per-LOS mean:

$$
\delta_F \to \delta_F - \langle \delta_F \rangle_{\rm LOS}
$$

### Why this is exact

The line-of-sight DFT at mode $k_n$ is:

$$
\tilde{\delta}(k_n) = \sum_{\alpha=0}^{N-1} \delta(\chi_\alpha) \, e^{i k_n \chi_\alpha}
$$

Subtracting the mean replaces $\delta \to \delta - \bar{\delta}$, where $\bar{\delta} = \frac{1}{N}\sum_\alpha \delta_\alpha$.
The correction term is:

$$
\sum_\alpha \bar{\delta} \, e^{i k_n \chi_\alpha} = \bar{\delta} \sum_\alpha e^{i k_n \chi_\alpha}
$$

For $k_n \neq 0$ on the FFT grid, the sum $\sum_\alpha e^{i k_n \chi_\alpha} = 0$
(geometric series / Dirichlet kernel), so **the correction vanishes identically**.
Only $k_0 = 0$ is affected.

### Test result

In [ ]:
dm = np.load(RESULTS_MARG)
cl_std = dm['cl_k_std']    # (Nsims, Nk, Nl)
cl_msub = dm['cl_k_msub']  # (Nsims, Nk, Nl)
k_par_m = dm['k_par']

Nsims_m, Nk_m, Nl_m = cl_std.shape
print(f'{Nsims_m} sims, {Nk_m} k-modes, {Nl_m} multipoles')
print(f'k_par = {k_par_m}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: k_par=0 — show that mean-sub kills the signal
ax = axes[0]
ells_m = np.arange(Nl_m)
std_mean_k0 = np.mean(cl_std[:, 0, :], axis=0)
msub_mean_k0 = np.mean(cl_msub[:, 0, :], axis=0)

ax.semilogy(ells_m[1:], std_mean_k0[1:], 'b-', lw=2, label='Standard $\\delta$')
ax.semilogy(ells_m[1:], np.abs(msub_mean_k0[1:]), 'r--', lw=2,
            label=r'$\delta - \langle\delta\rangle_{\rm LOS}$')
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$C_\ell(k_\parallel=0)$')
ax.set_title(r'$k_\parallel = 0$: marginalized away')
ax.legend(fontsize=14)

# Right: k_par≠0 — show perfect agreement
ax = axes[1]
for ik in range(1, Nk_m):
    std_mean = np.mean(cl_std[:, ik, :], axis=0)
    msub_mean = np.mean(cl_msub[:, ik, :], axis=0)
    valid = std_mean > 0
    frac_diff = np.abs(msub_mean[valid] / std_mean[valid] - 1)
    ax.semilogy(ells_m[valid], frac_diff, '-', lw=1.5,
                label=f'$k_\\parallel$={k_par_m[ik]:.4f}')

ax.axhline(1e-14, color='gray', ls=':', lw=1, label='Machine $\\epsilon$')
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$|C_\ell^{\rm msub}/C_\ell^{\rm std} - 1|$')
ax.set_title(r'$k_\parallel \neq 0$: unchanged (machine precision)')
ax.legend(fontsize=12)
ax.set_ylim(1e-16, 1e-10)

plt.tight_layout()
plt.show()

## 8. Recipe: Marginalizing Systematic Templates

### A. $k_\parallel$ marginalization (continuum fitting)

**Problem:** Continuum-fitting errors contribute a constant offset per sightline → they
live in the $k_\parallel=0$ Fourier mode.

**Solution:** Simply subtract the per-LOS mean before the Fourier transform:
```python
delta_F = delta_F - np.mean(delta_F, axis=1, keepdims=True)
```
Then exclude $k_\parallel = 0$ from the analysis. As demonstrated above, all $k \neq 0$
modes are numerically unchanged (to machine precision).

More generally, to marginalize over any LOS template $t(\chi)$ (e.g., a slope for
continuum tilt), project it out per sightline:
$$
\delta_F \to \delta_F - \frac{\langle \delta_F \cdot t \rangle}{\langle t \cdot t \rangle} \, t
$$
This removes power from all $k$ modes that have nonzero overlap with $t(\chi)$.

### B. $k_\perp$ marginalization (angular monopole)

**Problem:** Large-scale angular systematics (e.g., stellar contamination, depth variations)
affect the $\ell \approx 0$ modes, i.e. $k_\perp \approx 0$.

**Solution:** In the sFB framework, $k_\perp = (\ell + 1/2) / \chi_{\rm eff}$, so
$k_\perp = 0$ corresponds to $\ell = 0$. The monopole $C_{\ell=0}$ is already
unreliable for partial-sky surveys (and equals $N_{\rm skew}/(4\pi)$ for the uniform case).

Practical prescription:
1. **Exclude low-$\ell$ bins** from the analysis (e.g., start at $\ell_{\rm min} = $ NperBin).
   This is already done in the ratio tables above ("excl first bin").
2. **Project out an angular template:** To marginalize a template $T_\ell$ in the data vector,
   add it to the covariance: $C \to C + \lambda \, T T^T$ with $\lambda \to \infty$.
   In practice, add a large constant to the diagonal of the $\ell=0$ bin.

### C. Combined recipe

For a Ly-$\alpha$ P3D measurement:
1. Mean-subtract each LOS: `delta -= delta.mean(axis=1, keepdims=True)` → removes $k_\parallel=0$
2. Drop the first $\ell$-bin → removes $k_\perp \approx 0$
3. Optionally project out higher-order LOS templates (slope, quadratic) for additional
   continuum systematics
4. The remaining $(\ell, k)$ modes are unaffected by these operations

## 9. Per-Simulation Scatter

Show individual simulation measurements to illustrate the expected variance.

In [ ]:
# Show individual sim C_ell at k=0 and k=4 to illustrate scatter
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
NperBin_n = dn['NperBin']
Nl_n = dn['Nl']
Nbins_n = len(dn['binned_ells'])
Bn = make_binning_matrix(Nbins_n, NperBin_n, Nl_n)

for ax, ik, k_label in zip(axes, [0, 4],
                            [f"$k_\\parallel=0$",
                             f"$k_\\parallel={dn['k_par'][4]:.3f}$"]):
    for isim in range(min(dn['Nsims'], 10)):
        binned = Bn @ dn['cl_k_all'][isim, ik]
        ax.plot(dn['binned_ells'], binned, '-', alpha=0.3, lw=0.8)

    # Mean and theory
    binned_mean = Bn @ dn['cl_mean_all'][ik]
    binned_theory = Bn @ dn['theory_pseudo_all'][ik]
    ax.plot(dn['binned_ells'], binned_mean, 'k-', lw=2.5, label='Mean')
    ax.plot(dn['binned_ells'], binned_theory, 'r--', lw=2, label='Theory')
    ax.set_xlabel(r'$\ell$')
    ax.set_ylabel(r'pseudo-$C_\ell(k)$')
    ax.set_title(k_label)
    ax.legend(fontsize=14)

plt.suptitle(f'Individual simulations (thin) vs mean (black), {dn["Nsims"]} sims',
             fontsize=16)
plt.tight_layout()
plt.show()

## 10. Summary

| Test | Result |
|---|---|
| Multi-$k_\parallel$ recovery (10 modes) | All ratios within 1–3% of unity |
| RSD Kaiser factor | Correctly recovered at all $k_\parallel$ |
| Noise bias subtraction ($\sigma_c=0.10$) | Identical to noiseless after subtraction |
| $k_\parallel=0$ marginalization | Removed to $10^{-30}$; $k\neq 0$ unchanged to $10^{-14}$ |

The sFB $C_\ell(k_\parallel)$ estimator demonstrates excellent performance across all tests,
providing a solid foundation for Ly-$\alpha$ forest 3D power spectrum measurements.